# C12_E03 — MPC mínimo con scipy.optimize

**Capítulo:** 12 — Estrategias avanzadas  
**Identificador:** `C12_E03`  
**Objetivo pedagógico:** Comprender el horizonte rolante, las restricciones y la función de coste de un MPC.  
**Librerías:** scipy.optimize, numpy, matplotlib

## Ejemplo industrial

Planta lineal con restricciones de actuador: MPC frente a perturbación o cambio de consigna.

---

> **Aviso de uso responsable.** Este notebook es didáctico. No es código de producción. Cualquier implementación real requiere validación independiente. Ver `docs/politica_uso_responsable.md`.

## 1. MPC mínimo con scipy.optimize

In [1]:
# Implementación didáctica de MPC sobre planta lineal escalar.
# Para evitar dependencia de cvxpy, resolvemos el QP con scipy.optimize.minimize.
import numpy as np
import matplotlib.pyplot as plt
import os
from scipy.optimize import minimize

# Planta discreta: y[k+1] = a*y[k] + b*u[k]   (FOPDT discretizado, sin retardo)
a, b = 0.9, 0.05
N = 100              # horizonte total de simulación
Np = 10              # horizonte de predicción
Nc = 5               # horizonte de control
SP = 1.0
u_min, u_max = 0.0, 2.0

def predict(u_seq, y0):
    """Predicción Np pasos a partir de la secuencia de control u_seq (Nc valores)."""
    y = np.zeros(Np)
    yk = y0
    for k in range(Np):
        uk = u_seq[min(k, Nc-1)]   # u se mantiene tras el horizonte de control
        yk = a*yk + b*uk
        y[k] = yk
    return y

def cost(u_seq, y0):
    y_pred = predict(u_seq, y0)
    err = np.sum((SP - y_pred)**2)
    move = np.sum(np.diff(u_seq)**2)
    return err + 0.05*move

bounds = [(u_min, u_max)]*Nc

## 2. Simulación de bucle rolante

In [2]:
y = 0.0
ys, us = [], []
u_seq = np.full(Nc, 0.5)
for k in range(N):
    res = minimize(cost, u_seq, args=(y,), method='SLSQP', bounds=bounds,
                   options=dict(ftol=1e-6, maxiter=50))
    u_apply = float(res.x[0])
    y = a*y + b*u_apply
    ys.append(y); us.append(u_apply)
    u_seq = np.r_[res.x[1:], res.x[-1]]
ys, us = np.array(ys), np.array(us)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
ax1.plot(ys); ax1.axhline(SP, ls='--', color='gray'); ax1.set_ylabel("y"); ax1.grid(alpha=0.3)
ax1.set_title("C12_E03 — MPC mínimo (SLSQP) sobre planta lineal con restricciones")
ax2.step(range(N), us, where='post'); ax2.set_xlabel("k"); ax2.set_ylabel("u")
ax2.set_ylim(-0.1, 2.2); ax2.grid(alpha=0.3)
out = '../../figures/cap12/fig_C12_F03_mpc.png'
os.makedirs(os.path.dirname(out), exist_ok=True)
fig.tight_layout(); fig.savefig(out, dpi=120); plt.show()

## 3. Interpretación

MPC resuelve, en cada paso, un problema de optimización sobre el horizonte Np con restricciones explícitas en u. Se aplica solo el primer valor de la secuencia óptima y se vuelve a optimizar (horizonte rolante). Esta implementación con SLSQP es didáctica y suficiente para ilustrar el concepto. Para problemas industriales reales se usan solvers QP especializados (cvxpy, qpOASES, OSQP) y formulaciones más completas.